In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon, MultiPolygon

## Generating Data by Landkreis
Joins the RWI_GEO_REDX data on Landkreis level together with geographic data to generate germany map

In [4]:
file_districts = "data/gemeinden_simplify20.geojson"
map_municp_process = gpd.read_file(file_districts)

# 1. Den 5-stelligen Landkreis-Code extrahieren
# Wir nehmen die ersten 5 Stellen vom 'RS'
map_municp_process['landkreis_id'] = map_municp_process['RS'].str[:9]

In [6]:
map_municp_process.head()

,ADE,GF,BSG,RS,AGS,SDV_RS,GEN,BEZ,IBZ,BEM,...,FK_S3,NUTS,RS_0,AGS_0,WSK,DEBKG_ID,destatis,wikipedia,geometry,landkreis_id
0,6,4,1,010010000000,01001000,010010000000,Flensburg,Stadt,60,kreisfrei,...,R,DEF01,010010000000,01001000,2008-01-01,DEBKGDL20000E5MA,"{'RS': '010010000000', 'area': 56.73, 'populat...",None,"POLYGON ((9.41266 54.82264, 9.41318 54.82124, ...",010010000
1,6,4,1,010020000000,01002000,010020000000,Kiel,Stadt,60,kreisfrei,...,R,DEF02,010020000000,01002000,2006-01-01,DEBKGDL20000004J,"{'RS': '010020000000', 'area': 118.65, 'popula...",None,"POLYGON ((10.16916 54.43138, 10.17023 54.43, 1...",010020000
2,6,4,1,010030000000,01003000,010030000000,Lübeck,Stadt,60,kreisfrei,...,R,DEF03,010030000000,01003000,2006-02-01,DEBKGDL20000DYMA,"{'RS': '010030000000', 'area': 214.19, 'popula...",None,"POLYGON ((10.87684 53.98737, 10.87884 53.98595...",010030000
3,6,4,1,010040000000,01004000,010040000000,Neumünster,Stadt,60,kreisfrei,...,R,DEF04,010040000000,01004000,1970-04-26,DEBKGDL20000E4SA,"{'RS': '010040000000', 'area': 71.66, 'populat...",None,"POLYGON ((9.99545 54.14972, 9.99713 54.14806, ...",010040000
4,6,4,1,010510011011,01051011,010510011011,Brunsbüttel,Stadt,61,--,...,R,DEF05,010510011011,01051011,2009-01-01,DEBKGDL20000E2IK,"{'RS': '010510011011', 'area': 65.21, 'populat...",None,"POLYGON ((9.16439 53.94509, 9.16706 53.94302, ...",010510011


In [ ]:
# 1. Define the path and specific sheet name
# The sheet names in this Excel file usually match the CSV names we discussed

file_path = "data/RWI-GEO-REDX_PUF_v16/RWIGEOREDX_APPURC_V16_PUF_YEAR_ABS.xlsx"
#file_path = "data/RWI-GEO-REDX_PUF_v16/RWIGEOREDX_HOUPURC_V16_PUF_YEAR_ABS.xlsx"
sheet = "Munic_RegionEff_abs_yearly" # Use Munic_ or LMR_ depending on your goal

# 2. Load with skiprows
# RWI files typically have 3-5 rows of title/info at the top of the data sheets
df_municp = pd.read_excel(file_path, sheet_name=sheet, skiprows=0)

# 3. Clean up (sometimes they have an empty first column or row)
df_municp = df_municp.dropna(how='all', axis=0).dropna(how='all', axis=1)

df_municp.iloc[:5]

In [ ]:
def calculate_cagr(df, start_year, end_year, prefix='pindex'):
    n_years = end_year - start_year
    start_col = f'{prefix}{start_year}'
    end_col = f'{prefix}{end_year}'
    
    # CAGR Formel: [(Endwert / Anfangswert) ^ (1/n)] - 1
    # Wir nutzen np.where um Division durch Null oder NaNs abzufangen
    return (df[end_col] / df[start_col])**(1/n_years) - 1

# 5 Jahre (2020-2025)
df_distr['NOBS_yield_5y'] = calculate_cagr(df_distr, 2020, 2025, prefix='NOBS')
# 10 Jahre (2015-2025)
df_distr['NOBS_yield_10y'] = calculate_cagr(df_distr, 2015, 2025, prefix='NOBS')
# Full Dataset (2008-2025)
df_distr['NOBS_yield_full'] = calculate_cagr(df_distr, 2008, 2025, prefix='NOBS')

# Year-to-Year Profit (YoY) für das aktuellste Jahr
df_distr['NOBS_yield_yoy'] = (df_distr['NOBS2025'] / df_distr['NOBS2024']) - 1

# 5 Jahre (2020-2025)
df_distr['yield_5y'] = calculate_cagr(df_distr, 2020, 2025)
# 10 Jahre (2015-2025)
df_distr['yield_10y'] = calculate_cagr(df_distr, 2015, 2025)
# Full Dataset (2008-2025)
df_distr['yield_full'] = calculate_cagr(df_distr, 2008, 2025)

# Year-to-Year Profit (YoY) für das aktuellste Jahr
df_distr['profit_yoy'] = (df_distr['pindex2025'] / df_distr['pindex2024']) - 1

In [ ]:
# Wir suchen Ausreißer basierend auf zwei Kriterien:
# 1. NOBS explodiert im Vergleich zum Vorjahr/Nachfolger
# 2. Der Preisindex macht einen unnatürlichen Sprung (> 15% in einem Jahr ist selten)

def detect_real_estate_outliers(df, year):
    # Relativer Sprung NOBS (im Vergleich zum Vorjahr)
    nobs_col = f'NOBS{year}'
    prev_nobs = f'NOBS{year-1}'
    
    # Flag 1: NOBS ist mehr als 5x so groß wie im Vorjahr
    df[f'outlier_nobs_{year}'] = df[nobs_col] > (df[prev_nobs] * 5)
    
    # Flag 2: Preisindex-Sprung (Year-over-Year)
    p_col = f'pindex{year}'
    p_prev = f'pindex{year-1}'
    df[f'pct_change_{year}'] = (df[p_col] / df[p_prev]) - 1
    
    # Alles über 20% Steigerung oder 20% Fall in einem Jahr ist verdächtig
    df[f'outlier_price_{year}'] = df[f'pct_change_{year}'].abs() > 0.20
    
    return df

years = range(2009, 2026)

for year in years:
    detect_real_estate_outliers(df_distr, year)
    bad_rows = df_distr[df_distr[f'outlier_nobs_{year}']]
    print(f"Gefundene Fehler in {year}: {len(bad_rows)}")
    bad_rows[['kid2019',f'NOBS{year}',f'pindex{year}' ]].head()


In [ ]:
landkreise_map = gpd.read_file('data/landkreise_dissolved.geojson')

# 3. Merge
merged = landkreise_map.merge(df_distr, left_on='landkreis_id', right_on='kid2019')


In [ ]:

# 5. Plot a specific year (e.g., 2023)
# Ensure the column we want to plot is numeric
col = 'pindex2010'
merged[target_year] = pd.to_numeric(merged[col], errors='coerce')

# 5. Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# Plot districts with NO data in a light gray
landkreise_map.plot(ax=ax, color='#eeeeee', edgecolor='white', linewidth=0.3)

# Plot the merged data
merged.dropna(subset=[target_year]).plot(
    column=target_year, 
    ax=ax, 
    legend=True, 
    cmap='plasma', # 'plasma' or 'YlOrRd' are great for prices
    legend_kwds={
        'label': f"{col}", 
        'orientation': "horizontal",
        'pad': 0.05,
        'shrink': 0.8
    },
    edgecolor='black',
    linewidth=0.1
)

ax.set_title(f"REDX {col} by District ({target_year[-4:]})", fontsize=16, pad=20)
ax.set_axis_off()
plt.show()

In [ ]:
df_distr[df_distr['kid2019']=='11000'][[ 'pindex2025', 'pindex2020', 'pindex2015']]

## Processing .geojson